# 01 — Ingest provider files with PLEB

**You will:** write a mapping JSON, run `pleb ingest`, inspect its reports and verify
that every expected TOA reached the canonical tree. Prerequisite: notebook 00.
An ingest exit code of zero does not establish completeness: inspect missing files,
duplicate source matches and the observed row inventory.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
mapping = dict(sources=[str(ROOT/'raw')], par_roots=[str(ROOT/'raw/par')],
    template_roots=[], pulsars=PULSARS,
    backends={name:dict(root=str(ROOT/'raw'/name),tim_glob='*.tim') for name in ['A','B']})
mapping_path = CONFIGS/'mapping.json'
mapping_path.write_text(json.dumps(mapping,indent=2))
print(mapping_path.read_text())
ingest_config = lab.write_config(CONFIGS/'01_ingest.toml', dict(
    ingest_mapping_file=str(mapping_path), ingest_output_dir=str(REPO/'data'),
    ingest_verify=True, ingest_commit_branch_name='raw_ingest',
    ingest_commit_base_branch='scaffold', ingest_commit_message='Ingest synthetic provider files'))

`backends` determines the canonical TIM filenames; `par_roots` selects the starting
models. A broad glob can ingest unintended files. In a real release document aliases,
competing models and source priority. Keep telescope-supplied clocks alongside the inputs.

In [ ]:
lab.command(CLI+['ingest','--config',ingest_config], log=ROOT/'logs/01_ingest.log')
# PLEB committed raw_ingest in an isolated worktree. Adopt that exact tracked tree
# in the main training worktree; this is a generated workspace, never a release.
lab.command(['git','-C',REPO,'restore','--source','raw_ingest','--staged','--worktree','--','data'])
lab.command(['git','-C',REPO,'switch','raw_ingest'])
assert not lab.command(['git','-C',REPO,'status','--porcelain']).strip()
print('Reports:', [p.name for p in (REPO/'data/ingest_reports').iterdir()])
lab.command(['git','-C',REPO,'log','--all','--oneline','--decorate','-5'])
ingested = lab.snapshot(REPO,'raw_ingest',ROOT/'exports/01_ingested')
counts = lab.inventory(ingested)
display(counts)
assert counts.physical.sum()==580 and counts.active.sum()==580
assert counts.extra_active_duplicates.sum()==4
lab.verify_hashes(ROOT/'raw', json.loads((ROOT/'raw_manifest.json').read_text())['sha256'])

In [ ]:
breakdown = pd.read_csv(REPO/'data/ingest_reports/ingest_pulsar_breakdown.csv')
display(breakdown)
assert not breakdown.missing_parfile.any() and not breakdown.missing_timfiles.any()
for psr in PULSARS:
    assert (ingested/psr/f'{psr}.par').exists()
    assert len(list((ingested/psr/'tims').glob('*.tim')))==2
print((ingested/PULSARS[0]/f'{PULSARS[0]}_all.tim').read_text())

**Your task:** trace one raw TIM through the mapping, canonical filename and INCLUDE
wrapper. Why are there 580 physical rows but only 576 distinct measurements?
For another experiment, use a new workspace; do not rerun ingest over a release.